In [2]:
import formulallm.formula as f

import io
import contextlib

from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent, FunctionCallingAgentWorker
from llama_index.llms.ollama import Ollama

In [4]:
llama3 = Ollama(model="llama3", base_url='http://localhost:11434', temperature=0.0, request_timeout=600)

In [5]:
def load(filePath: str) -> str:
    """Load the Formula DSL code from the file path"""
    code = f.load(filePath)
    return code

load_tool = FunctionTool.from_defaults(fn=load)

In [6]:
def solve():
    """Try to solve the partial model based on the domain constraints"""
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.solve("pm","1","Mapping.conforms")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

solve_tool = FunctionTool.from_defaults(fn=solve)

In [7]:
def extract(task_id: str):
    """Extract the solve result to the task with task_id.
    If the partial model is solvable, will return the solution.
    Else if the partial model is unsolvable, will return the least unsatisfied core conditions.
    The task_id is a string representing an integer which is incremented by 1 each time we run the solve command."""
    
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        try:
            f.extract(task_id, "0", "0")
        except Exception as e:
            pass

    message = stdout_buffer.getvalue()
    error_message = stderr_buffer.getvalue()

    return message + "\n" + error_message

extract_result_tool = FunctionTool.from_defaults(fn=extract)

In [8]:
react_agent = ReActAgent.from_tools(tools=[load_tool, solve_tool, extract_result_tool], llm=llama3, verbose=True)

In [15]:
prompt = """
Load the file from file path './data/MappingExample.4ml'.
Then solve the partial model call the extract tool.
If the partial model is unsolvable, the extract tool will return the unsat core terms.
Use your own natural language processing ability to explain why it is unsolvable based on the file content and the unsatisfiable core conditions.
You shouldn't need any tools for the explanation step!"""

In [16]:
response = react_agent.chat(prompt)

print(str(response))

> Running step 4277b5e9-a96f-4abb-b123-b270e5ce8639. Step input: 
Load the file from file path './data/MappingExample.4ml'.
Then solve the partial model call the extract tool.
If the partial model is unsolvable, the extract tool will return the unsat core terms.
Use your own natural language processing ability to explain why it is unsolvable based on the file content and the unsatisfiable core conditions.
You shouldn't need any tools for the explanation step!
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: load
Action Input: {'filePath': './data/MappingExample.4ml'}
(Compiled) MappingExample.4ml
0.03s.
Observation: domain Mapping
{
  Component ::= new (id: Integer, utilization: Real).
  Processor ::= new (id: Integer).
  Mapping   ::= new (c: Component, p: Processor).

  // The utilization must be > 50
  invalidUtilization :- c is Component, c.utilization <= 50.

  badMapping :- p is Processor, 
		s = sum(0.0, { c.util